# Spark integration

Run the LO2v2 pipeline on a local Spark JVM with two execution threads. Spark reads CSV, cleans values, calculates counter deltas with an ordered window, pivots the seven services by timestamp, and groups complete time windows. Partitioning by run and scenario colocates related data; exchanges in the physical plan show shuffles. Spark SQL counts the train, validation and test windows. The pipeline checks each feature against the Pandas reference.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent

run = subprocess.run([sys.executable, "-m", "spark.lo2_pipeline"], cwd=root,
                     capture_output=True, text=True, errors="replace")
if run.returncode:
    raise RuntimeError(run.stdout[-3000:] + run.stderr[-6000:])
result = root / "results"
summary = json.loads((result / "spark_summary.json").read_text())
print(summary["spark_version"], summary["master"])
display(pd.DataFrame(summary["counts"]))
print("Physical plan preview:")
print("\n".join(line[:220] for line in run.stdout.splitlines()[:15]))

4.1.3 local[2]


,split,windows
0,test,834
1,train,2516
2,validation,857


Physical plan preview:
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [run#18, test#20, window_start#436L, split#1473, error#1474, client__container_memory_rss__mean#1475, client__container_memory_rss__std#1476, client__container_memory_working_set_bytes__mean#1477, cl
      +- InMemoryRelation [run#18, test#20, window_start#436L, split#1473, error#1474, client__container_memory_rss__mean#1475, client__container_memory_rss__std#1476, client__container_memory_working_set_bytes__mean#147
            +- AdaptiveSparkPlan isFinalPlan=false
               +- Project [run#18, test#20, window_start#436L, map(keys: [LO2_run_1739613601,LO2_run_1739754001,LO2_run_1739894401,LO2_run_1740164402,LO2_run_1740304801,LO2_run_1740574801,LO2_run_1740024001,LO2_run_17404
                  +- Filter (samples#437L = 20)
                     +- HashAggregate(keys=[run#18, test#20, window_start#436L], functions=[count(1), avg(client_container_memory_rss#286), stddev_pop(client_c

## DataFrames, persistence and inference

A DataFrame describes transformations until an action requests results. The SQL collection materializes the cached features, which are reused for parity checking and inference. Spark ships the PyTorch checkpoint to Python workers using a broadcast variable. Each worker loads the checkpoint and scores Arrow-backed batches through `mapInPandas`. Compare against central inference to verify scores and alerts. Local mode demonstrates parallel work within one machine.

In [2]:
predictions = pd.read_csv(result / "spark_predictions.csv")
counts = {row["split"]: row["windows"] for row in summary["counts"]}
assert len(predictions) == counts["test"]
assert not predictions[["run", "test", "window_start"]].duplicated().any()
display(predictions.head())
print(f"{len(predictions)} held-out predictions; {predictions.alert.sum()} alerts")

,run,test,window_start,score,alert
0,LO2_run_1738661358,access_token_auth_header_error_401,1738662334,0.537818,False
1,LO2_run_1738661358,access_token_auth_header_error_401,1738662344,0.634720,True
2,LO2_run_1738661358,access_token_auth_header_error_401,1738662354,0.673946,True
3,LO2_run_1738661358,access_token_auth_header_error_401,1738662364,0.533174,False
4,LO2_run_1738661358,access_token_auth_header_error_401,1738662374,0.631965,True


834 held-out predictions; 168 alerts


## HDFS deployment

The same DataFrame readers and writers accept HDFS URIs. The NameNode tracks block locations and the namespace; DataNodes store replicated blocks. Spark partitions govern work scheduling and differ from HDFS block replication. A shared HDFS endpoint and configured workers are needed to validate storage replication and multi-worker behavior. The local mode results do not measure either of those properties.